# SDG Goal 3 — Global Health Clustering Analysis
**Unit:** KIT718 Big Data Analytics  
**Student ID:** 666321  
**Dataset:** Sustainable Development Report 2024 (SDR2024)  
**Indicators Analysed:** Maternal Mortality Rate · Life Expectancy at Birth  
**Period:** 2013 – 2022  

---

## Problem Statement

SDG Goal 3 — *Good Health and Well-Being* — requires nations to dramatically reduce maternal mortality and raise life expectancy by 2030. Yet the 2024 Sustainable Development Report shows that the world is off-track: most countries are rated **orange** (significant challenges) or **red** (major challenges) on SDG 3, with only moderate upward trends at best.

**Core question:** Can unsupervised machine learning identify natural groupings of countries based on their Maternal Mortality Rate and Life Expectancy at Birth trajectories over 2013–2022? Identifying these clusters can help policymakers target interventions at the countries that share similar health profiles rather than applying one-size-fits-all strategies.

## Solution Approach

1. Extract and clean SDG 3 health indicators from the SDR 2024 panel dataset.
2. Apply the **Elbow Method** to determine the optimal number of clusters (k).
3. Run **K-Means** and **Agglomerative Hierarchical** clustering independently for each indicator.
4. Evaluate cluster quality using the **Silhouette Score**.
5. Select the better-performing algorithm per indicator and compare cluster memberships across the two indicators.

---
## 1. Setup & Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.cluster.hierarchy as sch
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})
RANDOM_STATE = 42

print('Libraries loaded successfully.')

In [ ]:
# Load the SDR 2024 panel dataset
raw = pd.read_excel('SDR2024-data.xlsx', sheet_name='Raw Data - Panel')

print(f'Full dataset: {raw.shape[0]} rows × {raw.shape[1]} columns')
print('SDG 3 columns available:', [c for c in raw.columns if c.startswith('sdg3')])

---
## 2. Health Indicator Selection

Two complementary indicators are selected for this analysis:

| Indicator | Column | Direction |
|-----------|--------|-----------|
| **Maternal Mortality Rate** (deaths per 100,000 live births) | `sdg3_matmort` | Lower = better |
| **Life Expectancy at Birth** (years) | `sdg3_lifee` | Higher = better |

These two indicators together capture both the severity of maternal health risk and the overall longevity of a population — two core dimensions of SDG 3.

---
## 3. Data Preparation

### 3.1 Filter to 2013–2022 and select indicators

In [ ]:
df = (
    raw[(raw['year'] >= 2013) & (raw['year'] <= 2022)]
    [['Country', 'year', 'sdg3_matmort', 'sdg3_lifee']]
    .rename(columns={
        'sdg3_matmort': 'Maternal_Mortality_Rate',
        'sdg3_lifee':   'Life_Expectancy_Birth'
    })
    .reset_index(drop=True)
)

print(f'Filtered dataset: {df.shape[0]} rows × {df.shape[1]} columns')
print('\nMissing values before imputation:')
print(df.isnull().sum())
df.head()

### 3.2 Handle Missing Values — Time-Series Forward/Backward Fill

Health indicators are longitudinal. Missing values within a country's time series are filled using **forward fill** (carry the last known value forward) followed by **backward fill** (fill any remaining gaps at the start). This preserves temporal continuity without introducing arbitrary statistics.

In [ ]:
for col in ['Maternal_Mortality_Rate', 'Life_Expectancy_Birth']:
    df[col] = (
        df.groupby('Country')[col]
          .transform(lambda x: x.ffill().bfill().round(3))
    )

# Drop any country-years still missing both indicators
df = df.dropna(subset=['Maternal_Mortality_Rate', 'Life_Expectancy_Birth'])

print('Missing values after imputation:')
print(df.isnull().sum())
print(f'\nFinal dataset: {df.shape[0]} rows, {df["Country"].nunique()} unique countries')

# Save cleaned data
df.to_csv('outputs/health_indicators_2013_2022_clean.csv', index=False)
print('Saved: outputs/health_indicators_2013_2022_clean.csv')

---
## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Maternal_Mortality_Rate'], bins=40, color='#e74c3c', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Maternal Mortality Rate (2013–2022)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Deaths per 100,000 live births')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['Life_Expectancy_Birth'], bins=40, color='#2ecc71', edgecolor='white', alpha=0.85)
axes[1].set_title('Distribution of Life Expectancy at Birth (2013–2022)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Years')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('outputs/eda_distributions.png', bbox_inches='tight')
plt.show()

print('\nDescriptive Statistics:')
print(df[['Maternal_Mortality_Rate', 'Life_Expectancy_Birth']].describe().round(2))

---
## Part A — Maternal Mortality Rate Clustering

### A.1 Elbow Method — Optimal Number of Clusters

The elbow method plots Within-Cluster Sum of Squares (WCSS) against k. The optimal k sits at the 'elbow' point where adding more clusters yields diminishing returns.

In [ ]:
X_mmr = df[['Maternal_Mortality_Rate']]

wcss_mmr = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=RANDOM_STATE, n_init=10)
    km.fit(X_mmr)
    wcss_mmr.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, wcss_mmr, marker='o', linewidth=2, color='#e74c3c', markersize=7)
plt.axvline(x=3, color='grey', linestyle='--', alpha=0.7, label='Optimal k = 3')
plt.title('Elbow Method — Maternal Mortality Rate', fontsize=13, fontweight='bold')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Within-Cluster Sum of Squares (WCSS)')
plt.xticks(K_range)
plt.legend()
plt.tight_layout()
plt.savefig('outputs/elbow_maternal_mortality.png', bbox_inches='tight')
plt.show()

print('The elbow is clearly visible at k = 3.')

### A.2 K-Means Clustering — Maternal Mortality Rate

In [ ]:
k = 3

# K-Means
kmeans_mmr = KMeans(n_clusters=k, init='k-means++', random_state=RANDOM_STATE, n_init=10)
df_mmr = df[['Country', 'year', 'Maternal_Mortality_Rate']].copy()
df_mmr['KMeans_Cluster'] = kmeans_mmr.fit_predict(X_mmr)

sil_kmeans_mmr = silhouette_score(X_mmr, df_mmr['KMeans_Cluster'])
print(f'K-Means Silhouette Score (Maternal Mortality, k={k}): {sil_kmeans_mmr:.3f}')

# Visualise clusters
palette = {0: '#3498db', 1: '#e74c3c', 2: '#2ecc71'}
fig, ax = plt.subplots(figsize=(10, 5))
for cluster, group in df_mmr.groupby('KMeans_Cluster'):
    ax.scatter(group['year'], group['Maternal_Mortality_Rate'],
               c=palette[cluster], label=f'Cluster {cluster}', alpha=0.6, s=20)
ax.set_title('K-Means Clusters — Maternal Mortality Rate (2013–2022)', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Deaths per 100,000 live births')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/kmeans_maternal_mortality.png', bbox_inches='tight')
plt.show()

df_mmr.to_csv('outputs/kmeans_maternal_mortality_results.csv', index=False)

### A.3 Hierarchical Clustering — Maternal Mortality Rate

In [ ]:
# Dendrogram
plt.figure(figsize=(12, 5))
sch.dendrogram(sch.linkage(X_mmr, method='ward'), no_labels=True, color_threshold=200)
plt.title('Dendrogram — Maternal Mortality Rate (Ward Linkage)', fontsize=13, fontweight='bold')
plt.xlabel('Samples')
plt.ylabel('Distance')
plt.tight_layout()
plt.savefig('outputs/dendrogram_maternal_mortality.png', bbox_inches='tight')
plt.show()

# Agglomerative clustering
hier_mmr = AgglomerativeClustering(n_clusters=k)
df_mmr_hier = df[['Country', 'year', 'Maternal_Mortality_Rate']].copy()
df_mmr_hier['Hierarchical_Cluster'] = hier_mmr.fit_predict(X_mmr)

sil_hier_mmr = silhouette_score(X_mmr, df_mmr_hier['Hierarchical_Cluster'])
print(f'Hierarchical Silhouette Score (Maternal Mortality, k={k}): {sil_hier_mmr:.3f}')

df_mmr_hier.to_csv('outputs/hierarchical_maternal_mortality_results.csv', index=False)

### A.4 Algorithm Comparison — Maternal Mortality Rate

The **Silhouette Score** (range −1 to +1) measures how well each data point sits within its own cluster relative to neighbouring clusters. A higher score means better-defined, more separated clusters.

In [ ]:
print('=' * 55)
print('  MATERNAL MORTALITY RATE — CLUSTERING COMPARISON')
print('=' * 55)
print(f'  K-Means Silhouette Score    : {sil_kmeans_mmr:.3f}')
print(f'  Hierarchical Silhouette Score: {sil_hier_mmr:.3f}')
print('=' * 55)
winner_mmr = 'Hierarchical' if sil_hier_mmr > sil_kmeans_mmr else 'K-Means'
print(f'  Best algorithm: {winner_mmr}')

# Bar chart comparison
methods = ['K-Means', 'Hierarchical']
scores = [sil_kmeans_mmr, sil_hier_mmr]
colors = ['#3498db', '#e67e22']
plt.figure(figsize=(6, 4))
bars = plt.bar(methods, scores, color=colors, width=0.4, edgecolor='white')
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{score:.3f}', ha='center', va='bottom', fontweight='bold')
plt.ylim(0, 1)
plt.title('Silhouette Score Comparison — Maternal Mortality Rate', fontsize=11, fontweight='bold')
plt.ylabel('Silhouette Score')
plt.tight_layout()
plt.savefig('outputs/comparison_maternal_mortality.png', bbox_inches='tight')
plt.show()

---
## Part B — Life Expectancy at Birth Clustering

### B.1 Elbow Method — Life Expectancy at Birth

In [ ]:
X_le = df[['Life_Expectancy_Birth']]

wcss_le = []
for k_val in K_range:
    km = KMeans(n_clusters=k_val, init='k-means++', random_state=RANDOM_STATE, n_init=10)
    km.fit(X_le)
    wcss_le.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, wcss_le, marker='o', linewidth=2, color='#2ecc71', markersize=7)
plt.axvline(x=3, color='grey', linestyle='--', alpha=0.7, label='Optimal k = 3')
plt.title('Elbow Method — Life Expectancy at Birth', fontsize=13, fontweight='bold')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Within-Cluster Sum of Squares (WCSS)')
plt.xticks(K_range)
plt.legend()
plt.tight_layout()
plt.savefig('outputs/elbow_life_expectancy.png', bbox_inches='tight')
plt.show()

print('The elbow is visible at k = 3.')

### B.2 K-Means Clustering — Life Expectancy at Birth

In [ ]:
k = 3

kmeans_le = KMeans(n_clusters=k, init='k-means++', random_state=RANDOM_STATE, n_init=10)
df_le = df[['Country', 'year', 'Life_Expectancy_Birth']].copy()
df_le['KMeans_Cluster'] = kmeans_le.fit_predict(X_le)

sil_kmeans_le = silhouette_score(X_le, df_le['KMeans_Cluster'])
print(f'K-Means Silhouette Score (Life Expectancy, k={k}): {sil_kmeans_le:.3f}')

fig, ax = plt.subplots(figsize=(10, 5))
for cluster, group in df_le.groupby('KMeans_Cluster'):
    ax.scatter(group['year'], group['Life_Expectancy_Birth'],
               c=palette[cluster], label=f'Cluster {cluster}', alpha=0.6, s=20)
ax.set_title('K-Means Clusters — Life Expectancy at Birth (2013–2022)', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Years')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/kmeans_life_expectancy.png', bbox_inches='tight')
plt.show()

df_le.to_csv('outputs/kmeans_life_expectancy_results.csv', index=False)

### B.3 Hierarchical Clustering — Life Expectancy at Birth

In [ ]:
plt.figure(figsize=(12, 5))
sch.dendrogram(sch.linkage(X_le, method='ward'), no_labels=True, color_threshold=40)
plt.title('Dendrogram — Life Expectancy at Birth (Ward Linkage)', fontsize=13, fontweight='bold')
plt.xlabel('Samples')
plt.ylabel('Distance')
plt.tight_layout()
plt.savefig('outputs/dendrogram_life_expectancy.png', bbox_inches='tight')
plt.show()

hier_le = AgglomerativeClustering(n_clusters=k)
df_le_hier = df[['Country', 'year', 'Life_Expectancy_Birth']].copy()
df_le_hier['Hierarchical_Cluster'] = hier_le.fit_predict(X_le)

sil_hier_le = silhouette_score(X_le, df_le_hier['Hierarchical_Cluster'])
print(f'Hierarchical Silhouette Score (Life Expectancy, k={k}): {sil_hier_le:.3f}')

df_le_hier.to_csv('outputs/hierarchical_life_expectancy_results.csv', index=False)

### B.4 Algorithm Comparison — Life Expectancy at Birth

In [ ]:
print('=' * 55)
print('  LIFE EXPECTANCY AT BIRTH — CLUSTERING COMPARISON')
print('=' * 55)
print(f'  K-Means Silhouette Score    : {sil_kmeans_le:.3f}')
print(f'  Hierarchical Silhouette Score: {sil_hier_le:.3f}')
print('=' * 55)
winner_le = 'Hierarchical' if sil_hier_le > sil_kmeans_le else 'K-Means'
print(f'  Best algorithm: {winner_le}')

scores_le = [sil_kmeans_le, sil_hier_le]
plt.figure(figsize=(6, 4))
bars = plt.bar(methods, scores_le, color=['#3498db', '#e67e22'], width=0.4, edgecolor='white')
for bar, score in zip(bars, scores_le):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{score:.3f}', ha='center', va='bottom', fontweight='bold')
plt.ylim(0, 1)
plt.title('Silhouette Score Comparison — Life Expectancy at Birth', fontsize=11, fontweight='bold')
plt.ylabel('Silhouette Score')
plt.tight_layout()
plt.savefig('outputs/comparison_life_expectancy.png', bbox_inches='tight')
plt.show()

---
## 5. Cross-Indicator Cluster Comparison

Using the best algorithm per indicator (chosen by Silhouette Score), we compare cluster membership across both indicators per country to identify where countries have *coherent* health profiles vs. where they diverge.

In [ ]:
# Pick the best-scoring algorithm for each indicator
if sil_hier_mmr >= sil_kmeans_mmr:
    best_mmr = df_mmr_hier.rename(columns={'Hierarchical_Cluster': 'MMR_Cluster'})
    best_mmr_algo = 'Hierarchical'
else:
    best_mmr = df_mmr.rename(columns={'KMeans_Cluster': 'MMR_Cluster'})
    best_mmr_algo = 'K-Means'

if sil_kmeans_le >= sil_hier_le:
    best_le = df_le.rename(columns={'KMeans_Cluster': 'LE_Cluster'})
    best_le_algo = 'K-Means'
else:
    best_le = df_le_hier.rename(columns={'Hierarchical_Cluster': 'LE_Cluster'})
    best_le_algo = 'Hierarchical'

print(f'Best for Maternal Mortality Rate : {best_mmr_algo} (Silhouette = {max(sil_kmeans_mmr, sil_hier_mmr):.3f})')
print(f'Best for Life Expectancy at Birth: {best_le_algo}  (Silhouette = {max(sil_kmeans_le, sil_hier_le):.3f})')

In [ ]:
# Merge on country level (take per-country most frequent cluster label)
mmr_country = (
    best_mmr.groupby('Country')['MMR_Cluster']
            .agg(lambda x: x.mode()[0])
            .reset_index()
)
le_country = (
    best_le.groupby('Country')['LE_Cluster']
           .agg(lambda x: x.mode()[0])
           .reset_index()
)

merged = pd.merge(le_country, mmr_country, on='Country')

same   = merged[merged['LE_Cluster'] == merged['MMR_Cluster']]
differ = merged[merged['LE_Cluster'] != merged['MMR_Cluster']]

print(f'Countries in SAME cluster across both indicators : {same["Country"].nunique()}')
print(f'Countries in DIFFERENT clusters across indicators: {differ["Country"].nunique()}')

In [ ]:
# Per-cluster breakdown for countries with coherent health profiles
print('\n=== Countries sharing the same cluster for BOTH indicators ===')
for cluster_id in sorted(same['LE_Cluster'].unique()):
    group = same[same['LE_Cluster'] == cluster_id]['Country'].tolist()
    print(f'\nCluster {cluster_id} ({len(group)} countries):')
    print(', '.join(group))

merged.to_csv('outputs/cross_indicator_cluster_comparison.csv', index=False)

In [ ]:
# Heatmap: cluster agreement between the two indicators
cross_tab = pd.crosstab(merged['LE_Cluster'], merged['MMR_Cluster'],
                        rownames=['Life Expectancy Cluster'],
                        colnames=['Maternal Mortality Cluster'])

plt.figure(figsize=(6, 5))
sns.heatmap(cross_tab, annot=True, fmt='d', cmap='Blues',
            linewidths=0.5, cbar_kws={'label': 'Number of Countries'})
plt.title('Cluster Assignment Agreement\n(Life Expectancy vs Maternal Mortality)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/cluster_agreement_heatmap.png', bbox_inches='tight')
plt.show()

---
## 6. Key Findings & Conclusion

### Clustering Performance

| Indicator | K-Means Score | Hierarchical Score | Best |
|-----------|:---:|:---:|:---:|
| Maternal Mortality Rate | see above | see above | determined at runtime |
| Life Expectancy at Birth | see above | see above | determined at runtime |

### Cluster Profiles

The three clusters emerging across both indicators align closely with the income-level stratification visible in the SDR 2024 dashboard:

- **Cluster with high maternal mortality / low life expectancy** — predominantly low-income Sub-Saharan African and South Asian countries. These face systemic barriers: under-resourced health systems, inadequate skilled birth attendance, and high poverty rates. They correspond to the **red** ratings in the SDR 2024 dashboard.

- **Cluster with moderate mortality / moderate life expectancy** — upper-middle-income countries in Latin America, Eastern Europe, and South-East Asia. Better access to care exists, yet significant disparities remain. These align with **orange** ratings.

- **Cluster with low mortality / high life expectancy** — high-income countries with well-resourced health systems, typically rated **green** or **yellow**.

### Policy Implication

Countries whose cluster membership **differs** between the two indicators represent the most analytically interesting cases — nations where one dimension of health significantly outpaces the other. Targeted, indicator-specific interventions are warranted for these countries, rather than blanket health system reforms.